# FINS5557 Applied AI in Finance
## Week 2: Applications, Opportunities and Limitations of AI

**UNSW Business School · Hric & Lin, 2026**

---

This notebook accompanies the Week 2 seminar. Code blocks mirror the seminar notes exactly, extending each section with runnable implementations.

### Notebook Contents

| # | Section | Seminar origin | Key method |
|---|---|---|---|
| 1 | Sentiment Analysis — TF-IDF Baseline | §Analysis | TF-IDF, logistic regression, cosine similarity |
| 2 | Sentiment Analysis — FinBERT Pipeline | §Analysis (seminar code) | Transformer-based finance sentiment |
| 3 | Mean–Variance Portfolio Optimisation | §Advisory | Markowitz optimisation, scipy.optimize |
| 4 | Next-Best-Action Scoring | §Advisory | Expected-value NBA, collaborative filtering |
| 5 | Credit Default — XGBoost (seminar code) | §Prediction (seminar code) | Gradient boosting, class weighting |
| 6 | SHAP Waterfall — Adverse Action Explanation | §Prediction + GDPR Art. 22 | Shapley values, individual explanation |
| 7 | LSTM Gate Visualisation | §Prediction | Gate activations on credit loss series |
| 8 | Isolation Forest — Spoofing Detection | §Risk & Compliance | Anomaly scoring, AUSTRAC surveillance |
| 9 | PSI Model Monitoring (seminar code) | §AI Regulation (seminar code) | Population Stability Index, AIRC drift |
| 10 | RAG Retrieval (seminar code) | §Hallucinations (seminar code) | Cosine retrieval, regulatory knowledge base |
| 11 | Calibration + Kalai Lower Bound | §Hallucinations | ECE, reliability diagrams, IIV bound |
| 12 | Attention Mechanism Visualisation | §LLM and Agentic Solutions | Scaled dot-product attention, heatmap |
| 13 | LSTM EUR/USD FX Forecasting | §Prediction (LSTM worked example) | Walk-forward LSTM, PSI regime monitoring |

In [ ]:
# Install packages not pre-installed in standard Colab runtime
# Sections 2 (FinBERT) requires GPU runtime for reasonable speed:
#   Runtime > Change runtime type > T4 GPU
import subprocess, sys

for pkg in ['xgboost', 'shap']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.datasets import make_classification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from scipy.optimize import minimize
import xgboost as xgb
import shap

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'DejaVu Sans'
})
NAVY, RED, GREEN = '#003366', '#C8102E', '#2E7D32'
print('Environment ready.')

---
## 1. Sentiment Analysis — TF-IDF Baseline

The first application domain is **analytical AI**: converting unstructured financial text into structured signals. This cell implements the TF-IDF baseline. Section 2 implements the production-grade FinBERT pipeline from the seminar notes.

**Cosine similarity of term vectors:**
$$\text{cosine}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \cdot \|\mathbf{v}\|}$$

Applied scenario: ASX earnings call — FinBERT assigns cosine similarity ≈ 0.87 between two positive sentences and ≈ 0.21 between a positive and a negative sentence, enabling fine-grained sentiment decomposition.

In [ ]:
# Earnings statement sentiment classifier — TF-IDF + logistic regression baseline
# Production replacement: substitute TfidfVectorizer with FinBERT embeddings (Section 2)

corpus = [
    ("Revenue grew 15 percent driven by strong retail banking performance", 1),
    ("Net interest income increased substantially year on year", 1),
    ("Operating expenses declined due to efficiency programme", 1),
    ("Capital ratios remain well above regulatory minimums", 1),
    ("Customer deposits grew 8 percent reflecting improved brand trust", 1),
    ("Digital channel adoption accelerated to 72 percent of transactions", 1),
    ("Credit losses are expected to normalise in the second half", 0),
    ("Net interest margins remain under significant pressure", 0),
    ("Provisions for credit impairment increased materially", 0),
    ("Regulatory capital requirements have increased substantially", 0),
    ("Cost of funds rose sharply due to central bank rate increases", 0),
    ("The outlook remains challenging given macroeconomic headwinds", 0),
    ("Loan growth was modest reflecting subdued consumer confidence", 0),
    ("We anticipate further deterioration in credit quality this quarter", 0),
]
texts, labels = zip(*corpus)

vectoriser = TfidfVectorizer(ngram_range=(1, 2), max_features=200)
X_tfidf = vectoriser.fit_transform(texts).toarray()
clf = LogisticRegression(C=1.0, max_iter=500, random_state=42)
clf.fit(X_tfidf, labels)

test_sentences = [
    "Full year results exceeded expectations with strong mortgage origination volumes.",
    "The board notes rising impairment charges and tightening margins in business lending.",
    "Wealth management revenue increased 12 percent on elevated market activity.",
    "Compliance costs rose significantly in response to ASIC enforcement action.",
]

print(f"{'Sentence':<68} Sentiment   Score")
print("-" * 90)
for s in test_sentences:
    prob  = clf.predict_proba(vectoriser.transform([s]).toarray())[0][1]
    label = 'POSITIVE' if prob > 0.5 else 'NEGATIVE'
    print(f"{s[:65]+'...' if len(s)>65 else s:<68} {label:<12} {prob:.3f}")

# Cosine similarity between query and corpus — illustrates the embedding geometry
query = "revenue growth and profitability improvement"
all_vecs = TfidfVectorizer(ngram_range=(1,2)).fit_transform(list(texts) + [query])
sims = cosine_similarity(all_vecs[-1], all_vecs[:-1]).flatten()
top5 = sims.argsort()[::-1][:5]

fig, ax = plt.subplots(figsize=(10, 3))
from matplotlib.patches import Patch
ax.barh([texts[i][:52] for i in top5], sims[top5],
        color=[NAVY if labels[i] == 1 else RED for i in top5])
ax.set_xlabel('Cosine Similarity'); ax.set_title(f'Top-5 Corpus Matches: "{query}"')
ax.legend(handles=[Patch(color=NAVY, label='Positive'), Patch(color=RED, label='Negative')])
plt.tight_layout(); plt.show()

---
## 2. Sentiment Analysis — FinBERT Production Pipeline

**Source:** Seminar notes §Analysis — this is the production-grade pipeline from the lecture notes.

FinBERT [Araci, 2019] is a BERT model fine-tuned on financial text from Reuters, earnings call transcripts, and 10-K filings. It produces **contextualised embeddings** — the representation of a token depends on its full sentence context, enabling disambiguation that TF-IDF cannot achieve.

> GPU runtime accelerates inference but is not required. When the FinBERT model cannot be downloaded (no internet, memory limit, or Colab free tier), a keyword-based mock runs automatically and produces output of identical structure.

The pipeline corresponds to **DDF Station 1 → Station 2**: raw transcript text → structured sentiment signal.

In [ ]:
# Section 2: Sentiment Analysis — FinBERT Production Pipeline
# Tries to load ProsusAI/finbert (~440 MB); falls back to keyword mock automatically.

import subprocess, sys

def _try_install_transformers():
    try:
        import transformers  # noqa: F401
        return True
    except ImportError:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'torch'])
            return True
        except Exception:
            return False

USE_LIVE_FINBERT = _try_install_transformers()
_classifier = None

if USE_LIVE_FINBERT:
    try:
        from transformers import pipeline as hf_pipeline, AutoTokenizer, AutoModelForSequenceClassification
        _tok = AutoTokenizer.from_pretrained("ProsusAI/finbert")
        _mdl = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
        _classifier = hf_pipeline("sentiment-analysis", model=_mdl, tokenizer=_tok,
                                   top_k=None, truncation=True, max_length=512)
        print("FinBERT loaded (ProsusAI/finbert, ~440 MB).")
    except Exception as e:
        USE_LIVE_FINBERT = False
        print(f"FinBERT unavailable ({e}).")

if not USE_LIVE_FINBERT:
    print("Running keyword-based mock — same output structure as live FinBERT.")

def score_transcript(text: str) -> pd.DataFrame:
    sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 20]
    results = []

    if USE_LIVE_FINBERT and _classifier is not None:
        for sentence in sentences:
            scores = _classifier(sentence)[0]
            best = max(scores, key=lambda x: x['score'])
            results.append({'sentence': sentence, 'label': best['label'],
                            'confidence': round(best['score'], 4)})
    else:
        import random
        random.seed(42)
        POS = {'grew', 'growth', 'strong', 'increased', 'improved', 'exceeded', 'higher'}
        NEG = {'pressure', 'deteriorat', 'declin', 'impairment', 'challenging', 'loss', 'fell'}
        for sentence in sentences:
            lower = sentence.lower()
            pos_hits = sum(1 for w in POS if w in lower)
            neg_hits = sum(1 for w in NEG if w in lower)
            if pos_hits > neg_hits:
                label, conf = 'positive', round(0.72 + random.uniform(0, 0.18), 4)
            elif neg_hits > pos_hits:
                label, conf = 'negative', round(0.68 + random.uniform(0, 0.20), 4)
            else:
                label, conf = 'neutral', round(0.55 + random.uniform(0, 0.25), 4)
            results.append({'sentence': sentence, 'label': label, 'confidence': conf})

    return pd.DataFrame(results)

excerpt = (
    "Revenue grew 12 percent year-on-year, driven by strong performance in retail banking. "
    "However, net interest margins remain under pressure. "
    "Credit losses are expected to normalise in the second half."
)

scores_df = score_transcript(excerpt)
tone_dist = scores_df['label'].value_counts(normalize=True)
print(f"\nDocument tone:\n{tone_dist.to_string()}")
print("\nSentence-level results:")
print(scores_df.to_string(index=False))

color_map = {'positive': NAVY, 'negative': RED, 'neutral': GREEN}
fig, ax = plt.subplots(figsize=(10, 3))
colors = [color_map.get(l.lower(), 'grey') for l in scores_df['label']]
ax.barh(range(len(scores_df)), scores_df['confidence'], color=colors, alpha=0.85)
ax.set_yticks(range(len(scores_df)))
ax.set_yticklabels([s[:50] for s in scores_df['sentence']], fontsize=8)
ax.set_xlabel('Confidence')
ax.set_title('FinBERT Sentence-Level Sentiment' if USE_LIVE_FINBERT
             else 'Sentiment Pipeline (Mock — keyword heuristic)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=NAVY, label='Positive'), Patch(color=RED, label='Negative'),
                   Patch(color=GREEN, label='Neutral')])
plt.tight_layout(); plt.show()


---
## 3. Mean–Variance Portfolio Optimisation

**Source:** Seminar notes §Advisory — Markowitz (1952) mean–variance framework.

Robo-advisors automate portfolio construction by solving:
$$\min_{\mathbf{w}} \; \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w} \quad \text{s.t.} \quad \mathbf{w}^\top \boldsymbol{\mu} \geq r^*, \quad \mathbf{1}^\top\mathbf{w} = 1, \quad \mathbf{w} \geq \mathbf{0}$$

**Applied scenario from slides:** ASX 200 ETF ($\mu_1=9\%$, $\sigma_1=14\%$) and AGBond ETF ($\mu_2=4\%$, $\sigma_2=5\%$), correlation $\rho = -0.15$. Client target return $r^* = 8\%$. The AI layer replaces static estimates with LSTM-based dynamic forecasts.

In [ ]:
# Mean-variance portfolio optimisation — Markowitz (1952)
# Implements the robo-advisor rebalancing scenario from seminar §Advisory

# --- Asset universe: ASX 200 ETF + AGBond ETF ---
assets  = ['ASX200 ETF', 'AGBond ETF']
mu      = np.array([0.09, 0.04])          # expected annual returns
sigma   = np.array([0.14, 0.05])          # annual volatilities
rho     = -0.15                            # correlation
cov     = np.array([
    [sigma[0]**2,          rho * sigma[0] * sigma[1]],
    [rho * sigma[0] * sigma[1], sigma[1]**2          ]
])

def portfolio_stats(w, mu, cov):
    """Return (expected_return, volatility) for weight vector w."""
    ret = w @ mu
    vol = np.sqrt(w @ cov @ w)
    return ret, vol

def min_variance_for_target(target_return, mu, cov, n=2):
    """Find minimum-variance portfolio achieving at least target_return."""
    constraints = [
        {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
        {'type': 'ineq', 'fun': lambda w: w @ mu - target_return},
    ]
    bounds  = [(0, 1)] * n
    w0      = np.ones(n) / n
    result  = minimize(lambda w: w @ cov @ w, w0, method='SLSQP',
                       bounds=bounds, constraints=constraints)
    return result.x if result.success else None

# Solve for r* = 8% target
r_star = 0.08
w_opt  = min_variance_for_target(r_star, mu, cov)
ret_opt, vol_opt = portfolio_stats(w_opt, mu, cov)
print(f"Optimal weights: {assets[0]}={w_opt[0]:.1%}  {assets[1]}={w_opt[1]:.1%}")
print(f"Expected return: {ret_opt:.2%}  |  Portfolio volatility: {vol_opt:.2%}")

# Efficient frontier
targets = np.linspace(mu.min() + 0.001, mu.max() - 0.001, 60)
frontier_vol, frontier_ret, frontier_w = [], [], []
for t in targets:
    w = min_variance_for_target(t, mu, cov)
    if w is not None:
        r, v = portfolio_stats(w, mu, cov)
        frontier_ret.append(r); frontier_vol.append(v); frontier_w.append(w[0])

# LSTM scenario: rising rate signal shifts mu_bond down to 3.2%
mu_lstm  = np.array([0.09, 0.032])
w_lstm   = min_variance_for_target(r_star, mu_lstm, cov)
ret_l, vol_l = portfolio_stats(w_lstm, mu_lstm, cov)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Efficient frontier
axes[0].plot(frontier_vol, frontier_ret, color=NAVY, lw=2.5, label='Efficient frontier')
axes[0].scatter([vol_opt], [ret_opt], color=RED, s=100, zorder=5,
                label=f'Optimal (r*={r_star:.0%}): {w_opt[0]:.0%}/{w_opt[1]:.0%}')
axes[0].scatter([vol_l], [r_star], color=GREEN, s=100, zorder=5, marker='^',
                label=f'LSTM rebalance: {w_lstm[0]:.0%}/{w_lstm[1]:.0%}')
axes[0].set_xlabel('Portfolio Volatility $\\sigma_p$')
axes[0].set_ylabel('Expected Return $\\mu_p$')
axes[0].set_title('Markowitz Efficient Frontier\nASX200 ETF + AGBond ETF')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[0].legend(fontsize=9)

# Weight shift from static to LSTM-rebalanced
cats    = ['Static (historical mu)', 'LSTM rebalance (rising rate signal)']
equit_w = [w_opt[0], w_lstm[0]]
bond_w  = [w_opt[1], w_lstm[1]]
x = np.arange(2)
axes[1].bar(x, equit_w, label='ASX200 ETF (equities)', color=NAVY, alpha=0.85)
axes[1].bar(x, bond_w, bottom=equit_w, label='AGBond ETF (bonds)', color=GREEN, alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(cats, fontsize=9)
axes[1].set_ylabel('Portfolio Weight'); axes[1].set_title('Robo-Advisor: Static vs LSTM Rebalance')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
axes[1].legend()

plt.tight_layout(); plt.show()
print(f"\nLSTM rebalance shifts equities from {w_opt[0]:.0%} to {w_lstm[0]:.0%} "
      f"as mu_bond drops from 4.0% to 3.2% under the rising rate signal.")

---
## 4. Next-Best-Action (NBA) Recommendation

**Source:** Seminar notes §Advisory — NBA expected-value scoring.

$$a_i^* = \arg\max_j \; \hat{P}(\text{accept}_{ij}) \cdot v_j - c_j$$

**Applied scenario from slides:** CommBank-style family segment client — income AUD 95k, 3-year seasoned mortgage. Home loan refinance dominates at AUD 3,570 expected value. FoFA Pt 7.7A Statements of Advice required.

In [ ]:
# Next-Best-Action simulation with segment-level scoring
# Implements seminar §Advisory NBA framework

np.random.seed(7)
N_CLIENTS = 2000
PRODUCTS  = ['home_loan_refi', 'personal_loan', 'savings_account', 'credit_card', 'wealth_platform']
LTV  = {'home_loan_refi': 12000, 'personal_loan': 3500, 'savings_account': 800,
         'credit_card': 1200, 'wealth_platform': 8000}
COST = {'home_loan_refi': 150, 'personal_loan': 40, 'savings_account': 10,
         'credit_card': 25, 'wealth_platform': 80}

segments = np.random.choice(['young_professional', 'family', 'pre_retiree', 'sme_owner'],
                             size=N_CLIENTS, p=[0.3, 0.35, 0.25, 0.10])
income   = np.where(segments == 'pre_retiree', np.random.normal(95000, 25000, N_CLIENTS),
           np.where(segments == 'sme_owner',   np.random.normal(120000, 40000, N_CLIENTS),
                                               np.random.normal(65000, 20000, N_CLIENTS))).clip(20000, 250000)

clients = pd.DataFrame({'segment': segments, 'income': income})

def uptake_prob(row, product):
    base = {'home_loan_refi': 0.04, 'personal_loan': 0.08, 'savings_account': 0.15,
            'credit_card': 0.12, 'wealth_platform': 0.03}
    p = base[product]
    if product == 'home_loan_refi'  and row.segment == 'family':              p *= 3.5
    if product == 'wealth_platform' and row.segment == 'pre_retiree':         p *= 6.0
    if product == 'personal_loan'   and row.segment == 'young_professional':  p *= 2.0
    if product == 'savings_account' and row.income > 90000:                   p *= 1.8
    return min(p * (row.income / 80000) ** 0.4, 0.85)

for prod in PRODUCTS:
    clients[f'prob_{prod}'] = clients.apply(lambda r: uptake_prob(r, prod), axis=1)
    clients[f'nba_{prod}']  = clients[f'prob_{prod}'] * LTV[prod] - COST[prod]

nba_cols = [f'nba_{p}' for p in PRODUCTS]
clients['best_action'] = clients[nba_cols].idxmax(axis=1).str.replace('nba_', '')
clients['best_value']  = clients[nba_cols].max(axis=1)

# Worked example: CommBank family segment client from slides
client_example = pd.Series({'segment': 'family', 'income': 95000})
print("NBA worked example — family segment, income AUD 95,000:")
print(f"  {'Product':<22} P(accept)   LTV      Cost    NBA score")
print("  " + "-" * 65)
for prod in PRODUCTS:
    p   = uptake_prob(client_example, prod)
    nba = p * LTV[prod] - COST[prod]
    flag = '  << BEST ACTION' if prod == 'home_loan_refi' else ''
    print(f"  {prod:<22} {p:.3f}       ${LTV[prod]:>6,}   ${COST[prod]:>4}   ${nba:>7,.0f}{flag}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
action_counts = clients['best_action'].value_counts()
axes[0].bar(action_counts.index, action_counts.values, color=NAVY, alpha=0.85)
axes[0].set_xlabel('Product'); axes[0].set_ylabel('Clients recommended')
axes[0].set_title('NBA Distribution Across Client Base'); axes[0].tick_params(axis='x', rotation=20)

seg_action = clients.groupby(['segment', 'best_action']).size().unstack(fill_value=0)
seg_action.plot(kind='bar', ax=axes[1], width=0.7)
axes[1].set_xlabel('Client Segment'); axes[1].set_ylabel('Count')
axes[1].set_title('NBA by Segment'); axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title='Product', fontsize=8)

plt.tight_layout(); plt.show()
print(f"\nEstimated total expected value: AUD {clients['best_value'].sum():,.0f}")
print("Regulatory note: Each recommendation requires Statements of Advice under FoFA Pt 7.7A.")

---
## 5. Credit Default — XGBoost + SHAP

**Source:** Seminar notes §Prediction — this cell reproduces the XGBoost code from the seminar exactly, extended with the SHAP waterfall worked example from §Explainability.

**XGBoost additive ensemble:**
$$\hat{y}_i = \sum_{t=1}^{T} f_t(\mathbf{x}_i), \quad f_t \in \mathcal{F}$$

Each tree fitted to second-order pseudo-residuals; `scale_pos_weight` compensates for the ~8% default rate. SHAP attribution satisfies SR 11-7 explainability requirements.

In [ ]:
# XGBoost credit default model with SHAP feature attribution
# Source: FINS5557 Week 2 seminar notes §Prediction
# Requires: xgboost, shap, sklearn, pandas, numpy, matplotlib

np.random.seed(42)

# Synthetic credit data: 10 features, ~8% default rate
X_raw, y = make_classification(n_samples=5000, n_features=10, n_informative=6,
                                weights=[0.92, 0.08], random_state=42)
feature_names = ['income_ratio', 'ltv', 'dti', 'months_employed',
                 'credit_score', 'prior_defaults', 'age', 'loan_amount',
                 'utilisation', 'num_accounts']
X = pd.DataFrame(X_raw, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train gradient-boosted ensemble (seminar code)
model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                           scale_pos_weight=(y==0).sum()/(y==1).sum(),
                           eval_metric='auc', random_state=42, verbosity=0)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f"XGBoost AUC-ROC: {auc:.4f}")

# SHAP explanation — addresses SR 11-7 model explainability requirements
explainer = shap.TreeExplainer(model)
shap_vals = explainer.shap_values(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[0].plot(fpr, tpr, color=NAVY, lw=2, label=f'AUC = {auc:.3f}')
axes[0].plot([0,1],[0,1],'--', color='grey', lw=1)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — Credit Default Model'); axes[0].legend()

shap.summary_plot(shap_vals, X_test, plot_type='bar', show=False)
axes[1].set_title('SHAP Feature Importance (SR 11-7 Documentation)')
plt.tight_layout()
plt.savefig('week2_credit_xgboost.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. SHAP Waterfall — Individual Adverse Action Explanation

**Source:** Seminar notes §Explainability + GDPR Article 22 worked example from slides.

**SHAP Shapley value:**
$$\phi_j = \sum_{S \subseteq \mathcal{F} \setminus \{j\}} \frac{|S|!(|\mathcal{F}|-|S|-1)!}{|\mathcal{F}|!} \left[ f(S \cup \{j\}) - f(S) \right]$$

**Efficiency axiom:** $\sum_j \phi_j = f(\mathbf{x}) - E[f(\mathbf{X})]$ — attributions sum exactly to the prediction gap from the portfolio baseline. This makes them auditable for GDPR Article 22 adverse action notices.

In [ ]:
# SHAP waterfall plot — individual adverse action explanation
# Demonstrates GDPR Article 22 / SR 11-7 individual-level explainability
# Applied scenario from slides: prior defaults +19pp, LTV +14pp, DTI +11pp, etc.

# Build SHAP Explanation object for the highest-risk test applicant
expected_value = explainer.expected_value
highest_risk_idx = np.argmax(y_proba)   # most likely to default in test set

print(f"Applicant #{highest_risk_idx}: predicted PD = {y_proba[highest_risk_idx]:.3f}")
print(f"Portfolio baseline PD       = {expected_value:.3f}")
print(f"PD excess above baseline    = {y_proba[highest_risk_idx] - expected_value:.3f}")

# Individual SHAP decomposition
ind_shap = shap_vals[highest_risk_idx]
shap_series = pd.Series(ind_shap, index=feature_names).sort_values()

print("\nSHAP decomposition (adverse action notice content):")
print(f"  {'Feature':<20} SHAP value   Direction")
print("  " + "-" * 48)
for feat, val in shap_series.sort_values(ascending=False).items():
    direction = 'increases default risk' if val > 0 else 'reduces default risk'
    print(f"  {feat:<20} {val:>+.4f}     {direction}")
print(f"\n  Sum of SHAP values: {ind_shap.sum():.4f}")
print(f"  f(x) - E[f(X)]   : {y_proba[highest_risk_idx] - expected_value:.4f}")
print(f"  Efficiency check  : {'PASS' if abs(ind_shap.sum() - (y_proba[highest_risk_idx] - expected_value)) < 0.01 else 'FAIL'}")

# Waterfall-style bar chart (manual, as shap.plots.waterfall requires newer shap)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top_feats = shap_series.reindex(shap_series.abs().sort_values(ascending=False).index[:8])
colors = [RED if v > 0 else NAVY for v in top_feats]
axes[0].barh(top_feats.index, top_feats.values, color=colors, alpha=0.85)
axes[0].axvline(0, color='black', lw=1)
axes[0].set_xlabel('SHAP value (impact on predicted PD)')
axes[0].set_title(f'Individual SHAP Waterfall\nApplicant PD={y_proba[highest_risk_idx]:.3f} '
                  f'(baseline={expected_value:.3f})')
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color=RED, label='Increases PD'), Patch(color=NAVY, label='Reduces PD')])

# Score distribution with applicant highlighted
axes[1].hist(y_proba[y_test == 0], bins=30, alpha=0.5, color=NAVY, label='No default', density=True)
axes[1].hist(y_proba[y_test == 1], bins=30, alpha=0.5, color=RED,  label='Default',    density=True)
axes[1].axvline(y_proba[highest_risk_idx], color='orange', ls='--', lw=2,
                label=f'This applicant ({y_proba[highest_risk_idx]:.3f})')
axes[1].set_xlabel('Predicted Default Probability')
axes[1].set_title('Score Distribution — Applicant Position'); axes[1].legend()

plt.tight_layout(); plt.show()

# Generate GDPR-compliant adverse action notice
top3 = shap_series.sort_values(ascending=False).head(3)
print("\n--- GDPR Article 22 Adverse Action Notice (auto-generated) ---")
print("Your application was declined. The primary factors were:")
for i, (feat, val) in enumerate(top3.items(), 1):
    print(f"  {i}. {feat.replace('_',' ').title()}: contribution +{val:.3f} to default probability")
print("You have the right to request human review of this decision under GDPR Article 22.")

---
## 7. LSTM Gate Visualisation

**Source:** Seminar notes §Prediction — LSTM gate equations and credit loss rate forecasting.

The LSTM's ability to model multi-year credit cycles depends on its gates:
$$\mathbf{f}_t = \sigma(\mathbf{W}_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f) \quad \text{(forget: what to discard)}$$
$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t \quad \text{(cell state: long-run memory)}$$

**Applied scenario from slides:** RBA cash rate forecasting — forget gate set to 0.1 when CPI drops sharply, input gate set to 0.9 on commodity shock. This cell numerically simulates gate dynamics on a synthetic credit loss rate series.

In [ ]:
# LSTM gate activation visualisation — numerical simulation
# Shows how forget/input/output gates respond to a macro shock on a credit loss rate series
# Source: FINS5557 Week 2 seminar notes §Prediction

def sigmoid(x):  return 1.0 / (1.0 + np.exp(-np.clip(x, -15, 15)))
def tanh_fn(x):  return np.tanh(np.clip(x, -15, 15))

np.random.seed(2026)
T = 60   # 5-year monthly series

# Synthetic credit loss rate: trend + 24-month cycle + macro shock at t=36
t    = np.arange(T)
clr  = 0.020 + 0.008 * np.sin(2 * np.pi * t / 24) + 0.0003 * t
clr[36:] += 0.012 * np.exp(-0.15 * np.arange(T - 36))  # shock decays
clr  = np.maximum(clr + np.random.normal(0, 0.0015, T), 0.005)

# Simulate scalar LSTM with hand-crafted weights illustrating gate behaviour
# Weights designed so the forget gate fires low (forget) when rate drops fast,
# and the input gate fires high when there is a sudden upward spike.
Wf, Uf, bf = 3.5,  -2.0,  -0.5   # forget gate: fires low when x drops, h_prev low
Wi, Ui, bi = 4.0,   1.0,  -1.5   # input gate:  fires high on upward spike
Wc, Uc, bc = 2.0,   0.5,   0.0   # cell candidate
Wo, Uo, bo = 1.5,   1.0,  -0.5   # output gate

h, c = 0.0, 0.0
f_vals, i_vals, o_vals, c_vals, h_vals = [], [], [], [], []

for xt in clr:
    # Normalise input around its mean for stability
    xt_n = (xt - 0.025) / 0.010
    f = sigmoid(Wf * xt_n + Uf * h + bf)
    i = sigmoid(Wi * xt_n + Ui * h + bi)
    c_cand = tanh_fn(Wc * xt_n + Uc * h + bc)
    c = f * c + i * c_cand
    o = sigmoid(Wo * xt_n + Uo * h + bo)
    h = o * tanh_fn(c)
    f_vals.append(f); i_vals.append(i); o_vals.append(o)
    c_vals.append(c); h_vals.append(h)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: CLR series
axes[0].plot(t, clr, color=NAVY, lw=2, label='Credit Loss Rate')
axes[0].axvline(36, color='orange', ls='--', lw=1.5, label='Macro shock (t=36)')
axes[0].set_ylabel('CLR'); axes[0].set_title('Credit Loss Rate — Input Series'); axes[0].legend(fontsize=9)

# Panel 2: gate activations
axes[1].plot(t, f_vals, color=RED,   lw=1.8, label='Forget gate $f_t$')
axes[1].plot(t, i_vals, color=NAVY,  lw=1.8, label='Input gate $i_t$')
axes[1].plot(t, o_vals, color=GREEN, lw=1.8, label='Output gate $o_t$')
axes[1].axvline(36, color='orange', ls='--', lw=1.5)
axes[1].set_ylabel('Gate activation [0,1]')
axes[1].set_title('LSTM Gate Activations')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1.05)

# Panel 3: cell state and hidden state
axes[2].plot(t, c_vals, color=NAVY,  lw=2, label='Cell state $c_t$ (long-run memory)')
axes[2].plot(t, h_vals, color=RED,   lw=1.5, ls='--', label='Hidden state $h_t$ (output)')
axes[2].axvline(36, color='orange', ls='--', lw=1.5, label='Macro shock')
axes[2].set_xlabel('Month'); axes[2].set_ylabel('State value')
axes[2].set_title('Cell State and Hidden State Dynamics'); axes[2].legend(fontsize=9)

plt.tight_layout(); plt.show()

# Explain the shock response at t=36
print(f"Gate values at macro shock (t=36):")
print(f"  Forget gate  f_36 = {f_vals[36]:.3f}  — {f_vals[36]:.0%} of prior cell state retained")
print(f"  Input gate   i_36 = {i_vals[36]:.3f}  — {i_vals[36]:.0%} of new information written")
print(f"  Output gate  o_36 = {o_vals[36]:.3f}")
print(f"  Cell state   c_36 = {c_vals[36]:.4f} (pre-shock: {c_vals[35]:.4f})")
print(f"\nVanilla RNN comparison: gradient of signal at t=36 after k steps back ≈ Wh^k")
print(f"  At k=10 steps: if |Wh|=0.9, gradient ≈ 0.9^10 = {0.9**10:.4f} (near-vanished)")
print(f"  LSTM cell state propagates the shock with forget gate = {f_vals[36]:.2f} per step")

---
## 8. Isolation Forest — Market Surveillance

**Source:** Seminar notes §Risk & Compliance — Isolation Forest anomaly score.

$$s(\mathbf{x}) = 2^{-\frac{E[h(\mathbf{x})]}{c(n)}}, \quad c(n) = 2H(n-1) - \frac{2(n-1)}{n}$$

- $h(\mathbf{x})$: average path length across isolation trees — anomalous points are isolated in few splits
- $s \to 1$: highly anomalous; $s \to 0.5$: normal

**Applied scenario from slides:** ASX spoofing detection — order cancellation rate 0.96, order-to-trade ratio 47:1, time-to-cancel 42 ms. Spoofing session scores 0.87 → AUSTRAC SAR triggered.

In [ ]:
# Isolation Forest market surveillance — spoofing and layering detection
# Source: FINS5557 Week 2 seminar notes §Risk & Compliance

np.random.seed(42)

N_NORMAL   = 4800
N_SPOOF    = 50     # spoofing sessions are rare
N_LAYERING = 30

# Features: [cancel_rate, order_trade_ratio, median_ttc_ms, volume_imbalance, session_duration_s]
# Normal trading session
normal = np.column_stack([
    np.random.beta(2, 8, N_NORMAL),                   # cancel rate ~0.2
    np.random.lognormal(1.5, 0.4, N_NORMAL),          # order:trade ~4-5:1
    np.random.lognormal(5.5, 0.5, N_NORMAL),          # TTC ~250 ms
    np.random.normal(0.0, 0.15, N_NORMAL),             # volume imbalance ~0
    np.random.uniform(300, 3600, N_NORMAL),            # session 5min-1hr
])

# Spoofing: high cancel rate, high OTR, very short TTC, positive volume imbalance
spoof = np.column_stack([
    np.random.beta(12, 1.5, N_SPOOF),                 # cancel rate ~0.88
    np.random.lognormal(3.8, 0.3, N_SPOOF),           # OTR ~45:1
    np.random.lognormal(3.7, 0.4, N_SPOOF),           # TTC ~42 ms
    np.random.uniform(0.6, 0.95, N_SPOOF),             # strong buy imbalance
    np.random.uniform(10, 120, N_SPOOF),               # short sessions
])

# Layering: moderate cancel rate, high OTR, alternating imbalance
layer = np.column_stack([
    np.random.beta(7, 2, N_LAYERING),
    np.random.lognormal(3.2, 0.3, N_LAYERING),
    np.random.lognormal(4.5, 0.4, N_LAYERING),
    np.abs(np.random.normal(0.5, 0.1, N_LAYERING)),
    np.random.uniform(60, 600, N_LAYERING),
])

X_all    = np.vstack([normal, spoof, layer])
labels   = (['normal'] * N_NORMAL + ['spoofing'] * N_SPOOF + ['layering'] * N_LAYERING)
feat_names = ['cancel_rate', 'order_trade_ratio', 'ttc_ms', 'vol_imbalance', 'session_s']

# Fit Isolation Forest — contamination matches known anomaly rate
clf_if = IsolationForest(n_estimators=200, contamination=(N_SPOOF + N_LAYERING) / len(labels),
                          random_state=42)
clf_if.fit(X_all)

# Anomaly scores: Isolation Forest returns -decision_function; rescale to [0,1]
raw_scores = clf_if.decision_function(X_all)
# Rescale: more negative = more anomalous → map to higher score
anom_scores = 1 - (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())
pred_labels = clf_if.predict(X_all)   # -1 = anomaly, 1 = normal

# --- Applied scenario: the spoofing session from the slides ---
asic_spoof = np.array([[0.96, 47.0, 42.0, 0.88, 30.0]])
asic_score = 1 - (clf_if.decision_function(asic_spoof)[0] - raw_scores.min()) / \
             (raw_scores.max() - raw_scores.min())
print(f"ASX spoofing session anomaly score: {asic_score:.3f}")
print(f"  cancel_rate=0.96, OTR=47:1, TTC=42ms, vol_imbalance=0.88")
if asic_score > 0.65:
    print("  >> Surveillance flag triggered — AUSTRAC SAR required within 3 business days")

# Confusion summary
from sklearn.metrics import classification_report
binary_true  = [0 if l == 'normal' else 1 for l in labels]
binary_pred  = [0 if p == 1 else 1 for p in pred_labels]
print("\nDetection performance:")
print(classification_report(binary_true, binary_pred, target_names=['Normal', 'Anomaly']))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distributions by true label
n_scores = anom_scores[:N_NORMAL]
s_scores = anom_scores[N_NORMAL:N_NORMAL+N_SPOOF]
l_scores = anom_scores[N_NORMAL+N_SPOOF:]

axes[0].hist(n_scores, bins=40, alpha=0.6, color=NAVY,  label=f'Normal (n={N_NORMAL})', density=True)
axes[0].hist(s_scores, bins=15, alpha=0.8, color=RED,   label=f'Spoofing (n={N_SPOOF})', density=True)
axes[0].hist(l_scores, bins=10, alpha=0.8, color='darkorange', label=f'Layering (n={N_LAYERING})', density=True)
axes[0].axvline(0.65, color='black', ls='--', lw=1.5, label='Flag threshold (0.65)')
axes[0].scatter([asic_score], [3], color='gold', s=150, zorder=6, label='ASIC scenario')
axes[0].set_xlabel('Anomaly Score'); axes[0].set_ylabel('Density')
axes[0].set_title('Isolation Forest Anomaly Score Distribution')
axes[0].legend(fontsize=8)

# Scatter: cancel_rate vs OTR coloured by anomaly score
sc = axes[1].scatter(X_all[:, 0], X_all[:, 1], c=anom_scores,
                     cmap='RdYlBu_r', s=8, alpha=0.7, vmin=0, vmax=1)
axes[1].scatter(asic_spoof[0, 0], asic_spoof[0, 1],
                color='gold', s=200, zorder=6, marker='*', label='ASIC scenario')
plt.colorbar(sc, ax=axes[1], label='Anomaly Score')
axes[1].set_xlabel('Cancel Rate'); axes[1].set_ylabel('Order-to-Trade Ratio')
axes[1].set_title('Feature Space — Anomaly Scores\n(high score = high suspicion)')
axes[1].legend()

plt.tight_layout(); plt.show()

---
## 9. Model Monitoring — Population Stability Index (PSI)

**Source:** Seminar notes §AI Regulation and Controls — this cell reproduces the PSI code from the seminar exactly.

$$\text{PSI} = \sum_{k=1}^{K} \left(p_k^{\text{current}} - p_k^{\text{baseline}}\right) \ln\!\left(\frac{p_k^{\text{current}}}{p_k^{\text{baseline}}}\right)$$

**PSI > 0.20 triggers AIRC review under SR 11-7 guidance.** The KL-divergence structure means each term is the information gain from observing the current bucket proportion relative to the baseline — larger when the shift is both large and concentrated in few buckets.

In [ ]:
# Population Stability Index (PSI) — detects model drift over time
# PSI > 0.20 triggers AIRC review under SR 11-7 guidance
# Source: FINS5557 Week 2 seminar notes §AI Regulation and Controls

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def compute_psi(expected: np.ndarray, actual: np.ndarray,
                n_bins: int = 10) -> float:
    """
    Compute Population Stability Index.
    expected: score distribution at training/baseline time
    actual:   score distribution at monitoring period
    Returns PSI value; PSI > 0.20 indicates significant drift.
    """
    bins = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    bins[0] -= 1e-9; bins[-1] += 1e-9

    exp_counts = np.histogram(expected, bins=bins)[0]
    act_counts = np.histogram(actual,   bins=bins)[0]

    exp_pct = np.where(exp_counts == 0, 1e-6, exp_counts / len(expected))
    act_pct = np.where(act_counts == 0, 1e-6, act_counts / len(actual))

    psi = np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))
    return psi

np.random.seed(99)

# Simulate score drift over 12 monthly monitoring windows
scores_baseline = np.random.beta(2, 5, 10000)       # training distribution
psi_values = []
months = list(range(1, 13))

for m in months:
    drift = m * 0.015          # gradual population shift
    scores_current = np.random.beta(2 + drift, 5 - drift * 0.5, 2000)
    psi_values.append(compute_psi(scores_baseline, scores_current))

plt.figure(figsize=(10, 4))
plt.plot(months, psi_values, marker='o', color='navy', lw=2)
plt.axhline(0.10, color='orange', ls='--', label='Warning (PSI=0.10)')
plt.axhline(0.20, color='red',    ls='--', label='Review trigger (PSI=0.20)')
plt.fill_between(months, psi_values,
                 where=[p > 0.20 for p in psi_values],
                 alpha=0.2, color='red')
plt.xlabel('Month'); plt.ylabel('PSI')
plt.title('Population Stability Index — Monthly Model Monitoring')
plt.legend(); plt.tight_layout()
plt.savefig('week2_psi_monitoring.png', dpi=150, bbox_inches='tight')
plt.show()

for m, psi in zip(months, psi_values):
    flag = ' *** AIRC REVIEW' if psi > 0.20 else (' * Warning' if psi > 0.10 else '')
    print(f"Month {m:2d}: PSI = {psi:.4f}{flag}")

# Bucket-level decomposition for Month 8 (illustrating how PSI accumulates)
drift = 8 * 0.015
scores_m8 = np.random.beta(2 + drift, 5 - drift * 0.5, 2000)
bins_eg = np.percentile(scores_baseline, np.linspace(0, 100, 6))
bins_eg[0] -= 1e-9; bins_eg[-1] += 1e-9
exp_c = np.histogram(scores_baseline, bins=bins_eg)[0]
act_c = np.histogram(scores_m8,       bins=bins_eg)[0]
exp_p = exp_c / len(scores_baseline); act_p = act_c / len(scores_m8)
contributions = (act_p - exp_p) * np.log(act_p / np.where(exp_p == 0, 1e-6, exp_p))
print(f"\nPSI bucket decomposition — Month 8 (total PSI={contributions.sum():.4f}):")
print(f"  {'Bucket':<20} Baseline%  Current%  Contribution")
for k in range(5):
    print(f"  [{bins_eg[k]:.3f} – {bins_eg[k+1]:.3f}]   "
          f"{exp_p[k]:.1%}      {act_p[k]:.1%}     {contributions[k]:+.4f}")

---
## 10. Retrieval-Augmented Generation (RAG)

**Source:** Seminar notes §Hallucinations — this cell reproduces the RAG code from the seminar exactly.

RAG grounds LLM generation in retrieved passages from a curated knowledge base, reducing hallucination on factual queries. The **Kalai et al. (2025) lower bound** establishes that RAG does not eliminate hallucination — it raises the information density in the context window above the singleton threshold, thereby reducing `err_IIV` for the specific query domain.

Production systems replace TF-IDF with dense semantic embeddings (`text-embedding-3-small`) in a vector database (Pinecone, pgvector). The retrieval logic is architecturally identical.

In [ ]:
# Retrieval-Augmented Generation (RAG) simulation with cosine similarity retrieval
# Demonstrates the information retrieval layer that grounds LLM responses
# Source: FINS5557 Week 2 seminar notes §Hallucinations

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import textwrap

# Curated regulatory knowledge base (stand-in for a vector store)
KNOWLEDGE_BASE = [
    "SR 11-7 requires banks to maintain a model inventory with conceptual soundness documentation.",
    "APRA CPS 234 mandates information security controls for all data assets classified as material.",
    "BCBS 239 requires risk data aggregation capabilities to produce consolidated risk reports within one business day.",
    "The EU AI Act classifies creditworthiness assessment AI systems as high-risk under Annex III.",
    "GDPR Article 22 gives individuals the right to not be subject to solely automated decisions with legal effects.",
    "Population Stability Index above 0.20 indicates significant distributional shift requiring model review.",
    "AUSTRAC requires reporting entities to submit Suspicious Matter Reports within three business days.",
    "MiFID II requires algorithmic trading firms to maintain kill switches and circuit breakers.",
    "SMOTE generates synthetic minority class samples by interpolation between existing instances.",
    "XGBoost uses second-order gradient statistics to fit trees to pseudo-residuals of the ensemble.",
]

def rag_retrieve(query: str, knowledge_base: list, top_k: int = 3) -> list:
    """Retrieve top-k most relevant passages using TF-IDF cosine similarity."""
    corpus = knowledge_base + [query]
    vectorizer   = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(corpus)

    query_vec = tfidf_matrix[-1]
    doc_vecs  = tfidf_matrix[:-1]
    sims      = cosine_similarity(query_vec, doc_vecs).flatten()
    top_idx   = sims.argsort()[::-1][:top_k]

    return [(knowledge_base[i], round(float(sims[i]), 4)) for i in top_idx]

# Simulate queries a compliance analyst might pose
queries = [
    "What PSI threshold triggers model review?",
    "What are the EU AI Act requirements for credit scoring?",
    "How quickly must suspicious transactions be reported?",
]

for q in queries:
    print(f"\nQuery: {q}")
    results = rag_retrieve(q, KNOWLEDGE_BASE)
    for i, (passage, score) in enumerate(results, 1):
        wrapped = textwrap.fill(passage, width=70, initial_indent='   ')
        print(f"  [{i}] (sim={score:.4f}) {wrapped}")

---
## 11. Calibration + Kalai et al. Lower Bound

**Source:** Seminar notes §Hallucinations — IIV lower bound and calibration analysis.

**Kalai et al. (2025) lower bound:**
$$\text{err}_G \geq 2 \cdot \text{err}_{\text{IIV}} - \frac{\max_c |V_c|}{\min_c |E_c|} - \delta$$

**Expected Calibration Error:**
$$\text{ECE} = \sum_{k=1}^{K} \frac{|B_k|}{N} \left| \text{acc}(B_k) - \text{conf}(B_k) \right|$$

**Applied scenario from slides:** RLHF alignment raises ECE from 0.043 to 0.104 — the model becomes 2.4× more miscalibrated; a compliance officer acting on 90%+ confidence responses would be wrong 21% of the time.

In [ ]:
# Hallucination calibration analysis + Kalai et al. (2025) lower bound visualisation
# Source: FINS5557 Week 2 seminar notes §Hallucinations

np.random.seed(2026)
N = 3000

# Ground truth distribution
true_prob   = np.random.beta(1.5, 3.0, N)
true_labels = np.random.binomial(1, true_prob)

# Three calibration profiles:
# 1. Well-calibrated (pre-RLHF)
well_cal = np.clip(true_prob + np.random.normal(0, 0.06, N), 0.01, 0.99)

# 2. Over-confident (post-RLHF: annotators reward confident answers)
over_conf = np.where(well_cal > 0.5,
                     np.clip(well_cal * 1.4 + 0.15, 0.01, 0.99),
                     np.clip(well_cal * 0.6 - 0.10, 0.01, 0.99))

# 3. Calibrated + abstention (RAG + confidence threshold)
ABSTAIN_THRESH = 0.35
abstain_mask   = (well_cal > ABSTAIN_THRESH) & (well_cal < (1 - ABSTAIN_THRESH))

def expected_calibration_error(y_true, y_prob, n_bins=10):
    """Expected Calibration Error — mean |confidence - accuracy| weighted by bin size."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0: continue
        ece += mask.sum() / len(y_true) * abs(y_true[mask].mean() - y_prob[mask].mean())
    return ece

non_abs = ~abstain_mask
ece_wc = expected_calibration_error(true_labels, well_cal)
ece_oc = expected_calibration_error(true_labels, over_conf)
ece_ca = expected_calibration_error(true_labels[non_abs], well_cal[non_abs])

print(f"ECE (well-calibrated pre-RLHF):    {ece_wc:.4f}")
print(f"ECE (over-confident post-RLHF):    {ece_oc:.4f}  << worse after RLHF alignment")
print(f"ECE (calibrated + abstention):     {ece_ca:.4f}  << improved by abstaining")
print(f"Abstention rate:                   {abstain_mask.mean():.1%}")
print(f"Calibration degradation factor:    {ece_oc/ece_wc:.1f}x worse after RLHF")

frac_wc, mean_wc = calibration_curve(true_labels, well_cal,  n_bins=10)
frac_oc, mean_oc = calibration_curve(true_labels, over_conf, n_bins=10)
frac_ca, mean_ca = calibration_curve(true_labels[non_abs], well_cal[non_abs], n_bins=8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reliability diagrams
axes[0].plot([0,1],[0,1], '--', color='grey', lw=1.5, label='Perfect calibration')
axes[0].plot(mean_wc, frac_wc, 'o-', color=NAVY,  lw=2, label=f'Pre-RLHF (ECE={ece_wc:.3f})')
axes[0].plot(mean_oc, frac_oc, 's--',color=RED,   lw=2, label=f'Post-RLHF (ECE={ece_oc:.3f})')
axes[0].plot(mean_ca, frac_ca, '^-', color=GREEN, lw=2, label=f'Abstention (ECE={ece_ca:.3f})')
axes[0].set_xlabel('Mean Predicted Probability'); axes[0].set_ylabel('Fraction Positive')
axes[0].set_title('Reliability Diagram — RLHF Calibration Effect')
axes[0].legend(fontsize=9)

# Kalai lower bound
iiv_err = np.linspace(0.01, 0.40, 200)
axes[1].plot(iiv_err, 2 * iiv_err, color=NAVY, lw=2.5,
             label='Lower bound: $\\text{err}_G \\geq 2 \\cdot \\text{err}_{IIV}$')
axes[1].fill_between(iiv_err, 2 * iiv_err, 0, alpha=0.10, color=NAVY)
# Mark specific scenarios
for iiv, label, color in [(0.05, 'Regulatory penalty clause\n(singleton, IIV=5%)', RED),
                           (0.01, 'BSM formula\n(well-attested, IIV=1%)', GREEN)]:
    axes[1].axvline(iiv, color=color, ls='--', lw=1.5)
    axes[1].scatter([iiv], [2 * iiv], color=color, s=100, zorder=5)
    axes[1].annotate(f'{label}\nerr_G ≥ {2*iiv:.0%}',
                     xy=(iiv, 2 * iiv), xytext=(iiv + 0.03, 2 * iiv + 0.04),
                     fontsize=8, color=color,
                     arrowprops=dict(arrowstyle='->', color=color))
axes[1].set_xlabel('IIV Misclassification Rate'); axes[1].set_ylabel('Min Generative Error Rate')
axes[1].set_title('Kalai et al. (2025) Lower Bound\n$\\text{err}_G \\geq 2 \\cdot \\text{err}_{IIV}$')
axes[1].legend(fontsize=9); axes[1].set_xlim(0, 0.40); axes[1].set_ylim(0, 0.85)

plt.tight_layout(); plt.show()

---
## 12. Attention Mechanism Visualisation

**Source:** Seminar notes §LLM and Agentic Solutions — scaled dot-product attention.

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right)\mathbf{V}$$

**Applied scenario from slides:** bond indenture clause resolution — query token "unless" attends with weight 0.72 to the key token "§12.1", enabling the model to resolve a cross-reference 400 tokens away — a task that would require 400 recurrent steps in an LSTM.

This cell simulates attention weights for a short financial sentence and visualises the resulting attention heatmap.

In [ ]:
# Scaled dot-product attention visualisation
# Source: FINS5557 Week 2 seminar notes §LLM and Agentic Solutions
# Simulates attention weights for a financial sentence — no PyTorch required

np.random.seed(42)

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    """Scaled dot-product attention (Vaswani et al., 2017)."""
    d_k = Q.shape[-1]
    scores  = Q @ K.T / np.sqrt(d_k)   # compatibility matrix [seq_len x seq_len]
    weights = softmax(scores, axis=-1)   # row-wise softmax -> attention distribution
    output  = weights @ V                # weighted sum of value vectors
    return output, weights

# --- Scenario A: bond indenture cross-reference resolution ---
# Tokens: [The, Issuer, shall, not, unless, See, §12.1, pledge, any, assets]
tokens_indenture = ['The', 'Issuer', 'shall', 'not', 'unless', 'See', '§12.1', 'pledge', 'any', 'assets']
n_tok, d_k = len(tokens_indenture), 8

# Craft embeddings so "unless" strongly attends to "§12.1"
np.random.seed(7)
X_base = np.random.randn(n_tok, d_k) * 0.5
# "unless" (idx 4) and "§12.1" (idx 6) are semantically linked — share a latent dimension
X_base[4, 0] = 2.5;  X_base[6, 0] = 2.5   # shared "conditional" dimension
X_base[4, 1] = 1.8;  X_base[6, 1] = 1.5
# "not" (idx 3) and "unless" (idx 4) share a negation dimension
X_base[3, 2] = 1.5;  X_base[4, 2] = 1.2

# Random learned projection matrices
Wq = np.random.randn(d_k, d_k) * 0.4
Wk = np.random.randn(d_k, d_k) * 0.4
Wv = np.random.randn(d_k, d_k) * 0.3

Q_mat = X_base @ Wq
K_mat = X_base @ Wk
V_mat = X_base @ Wv

output_indenture, attn_weights_indenture = scaled_dot_product_attention(Q_mat, K_mat, V_mat)

# --- Scenario B: earnings call — "provision" disambiguation ---
tokens_earn = ['Net', 'interest', 'income', 'declined', 'as', 'loan', 'loss', 'provisions', 'increased', 'sharply']
n_tok2 = len(tokens_earn)
X2 = np.random.randn(n_tok2, d_k) * 0.5
# "provisions" (idx 7) attends to "loan" (idx 5) and "loss" (idx 6)
for i in [5, 6, 7]: X2[i, 3] = 2.0
# "declined" (idx 3) attends to "income" (idx 2)
X2[2, 4] = 1.8; X2[3, 4] = 1.6

Q2 = X2 @ Wq; K2 = X2 @ Wk; V2 = X2 @ Wv
_, attn_weights_earn = scaled_dot_product_attention(Q2, K2, V2)

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, weights, tokens, title in [
    (axes[0], attn_weights_indenture, tokens_indenture,
     'Attention Heatmap — Bond Indenture\n"unless" → "§12.1" cross-reference'),
    (axes[1], attn_weights_earn, tokens_earn,
     'Attention Heatmap — Earnings Call\n"provisions" → "loan loss" disambiguation'),
]:
    im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=weights.max(), aspect='auto')
    ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens, fontsize=9)
    ax.set_xlabel('Key tokens (attended to)'); ax.set_ylabel('Query tokens (attending from)')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='Attention weight')

plt.tight_layout(); plt.show()

# Report key attention statistics for the indenture scenario
print("Bond indenture attention — 'unless' token (row 4) top-3 attended keys:")
unless_row = attn_weights_indenture[4]
top3_idx   = unless_row.argsort()[::-1][:3]
for rank, idx in enumerate(top3_idx, 1):
    print(f"  [{rank}] '{tokens_indenture[idx]:<8}' weight = {unless_row[idx]:.3f}")

print("\nVanilla RNN comparison:")
print("  To propagate the §12.1 signal to 'unless' (gap = 2 tokens here, up to 400 in full indenture),")
print("  an RNN gradient would decay as Wh^k. At |Wh|=0.9, k=400: gradient ≈", f"{0.9**400:.2e}")
print("  Attention resolves cross-references regardless of positional distance.")

print("\nNote: Full transformer uses multi-head attention (H parallel heads) + positional encoding.")
print("      Full derivation and pre-trained model inference covered in Week 3.")

---
## 13. LSTM EUR/USD FX Forecasting — Worked Example

**Source:** Week 2 slides — Session 1, slides 13–14. Companion to seminar notes §Prediction.

**Problem:** Predict the binary direction (up / down) of the next-day EUR/USD log return using a six-feature LSTM trained on 14 years of synthetic daily data calibrated to realistic distributional properties.

**Feature set:**

| Feature | Economic motivation |
|---|---|
| EUR/USD log return $r_t$ | Lagged momentum |
| Rate differential $\Delta i_t = i^{US}_t - i^{EU}_t$ | Carry signal — primary FX driver at medium horizons |
| VIX level | Risk-off/on regime indicator |
| 20-day realised volatility | Regime identification |
| RSI(14) | Momentum / mean-reversion |
| US 10Y–2Y term spread | Macro sentiment; recession signal |

**Architecture:** input (60, 6) → LSTM(64) → Dropout(0.2) → LSTM(32) → Dropout(0.2) → Dense(1, sigmoid)

**Data split:** train 2010–2021 → validation 2022 → test 2023–2024 (walk-forward, no look-ahead)

**Key result from slides:**

| Model | Directional accuracy | Sharpe (annualised) | Max drawdown |
|---|---|---|---|
| Random walk | 50.0% | −0.08 | −18.4% |
| ARIMA proxy | 51.2% | +0.11 | −15.7% |
| LSTM — price features only | 52.1% | +0.18 | −14.2% |
| **LSTM — full feature set** | **55.4%** | **+0.47** | **−9.8%** |

The 5.4 pp accuracy gain comes almost entirely from macro features (rate differential, VIX), consistent with Lopez de Prado (2020).

> **Governance link:** The rate differential shifts from ~0 bps (training era) to >400 bps (2023), producing PSI > 0.30 — a mandatory AIRC review trigger under SR 26-2 model monitoring practice.

> The implementation uses **sklearn MLPClassifier** (no TensorFlow required). The full pipeline — sequence construction, walk-forward evaluation, and PSI monitoring — is identical to the LSTM architecture described above; only the internal cell architecture differs.

In [ ]:
# Section 13: EUR/USD FX Direction Forecasting — Worked Example
# Source: FINS5557 Week 2 slides 13-14 (Session 1)
# Stack: numpy, pandas, sklearn, matplotlib (all standard Colab — no TensorFlow required)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

NAVY, RED, GREEN = "#003366", "#C8102E", "#2E7D32"
np.random.seed(2026)

# ── Synthetic EUR/USD dataset (calibrated to 2010-2024 distributional properties) ───
T = 3528
dates = pd.date_range("2010-01-04", periods=T, freq="B")

rate_diff = np.zeros(T)
rate_diff[:3000] = np.cumsum(np.random.normal(0, 0.003, 3000))
rate_diff[3000:] = rate_diff[2999] + np.cumsum(np.random.normal(0.005, 0.003, T - 3000))
rate_diff = np.clip(rate_diff, -0.02, 0.055)

vix = np.clip(15 + np.cumsum(np.random.normal(0, 0.45, T)), 10, 85)
vix[800:820]   += 22   # European debt crisis 2012
vix[2500:2515] += 38   # COVID March 2020
vix[3200:3210] += 20   # SVB March 2023

log_ret = np.random.normal(0, 0.004, T)
log_ret += 0.00015 * np.sign(rate_diff) * np.abs(rate_diff) * 50
log_ret += np.random.normal(0, 0.0025, T)

real_vol    = pd.Series(log_ret).rolling(20).std().fillna(0.004).values
rsi_raw     = np.clip(50 + 20 * np.cumsum(np.random.normal(0, 0.018, T)), 15, 85)
term_spread = np.clip(np.cumsum(np.random.normal(0, 0.003, T)) + 0.015, -0.01, 0.04)
eurusd_ret  = log_ret + np.random.normal(0, 0.0012, T)

FEATURES = ["log_return", "rate_diff", "vix", "real_vol", "rsi", "term_spread"]
df = pd.DataFrame({
    "log_return": eurusd_ret, "rate_diff": rate_diff,
    "vix": vix / 100,        "real_vol":  real_vol,
    "rsi": rsi_raw / 100,    "term_spread": term_spread,
}, index=dates)
df["target"] = (df["log_return"].shift(-1) > 0).astype(int)
df = df.dropna()

# ── Walk-forward split (no look-ahead) ────────────────────────────────────────
df_train = df[df.index <= "2021-12-31"]
df_test  = df[df.index > "2022-12-31"]
SEQ_LEN  = 60

def make_sequences(frame, feats, seq_len):
    arr, tgt = frame[feats].values, frame["target"].values
    X, y = [], []
    for i in range(seq_len, len(arr)):
        X.append(arr[i - seq_len:i])
        y.append(tgt[i])
    return np.array(X), np.array(y)

X_tr, y_tr = make_sequences(df_train, FEATURES, SEQ_LEN)
X_te, y_te = make_sequences(df_test,  FEATURES, SEQ_LEN)

sc = StandardScaler()
n  = len(FEATURES)
X_tr_sc = sc.fit_transform(X_tr.reshape(-1, n)).reshape(X_tr.shape)
X_te_sc = sc.transform(X_te.reshape(-1, n)).reshape(X_te.shape)

print(f"Train {X_tr_sc.shape} | Test {X_te_sc.shape}")

# ── Model A: MLP on full feature set ─────────────────────────────────────────
flat_tr = X_tr_sc.reshape(len(X_tr_sc), -1)
flat_te = X_te_sc.reshape(len(X_te_sc), -1)

model_full = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300,
                           random_state=42, early_stopping=True, validation_fraction=0.15)
model_full.fit(flat_tr, y_tr)
p_full   = model_full.predict_proba(flat_te)[:, 1]
acc_full = accuracy_score(y_te, (p_full > 0.5).astype(int))

# ── Model B: price-only baseline ─────────────────────────────────────────────
PRICE = ["log_return", "real_vol", "rsi"]
Xp_tr, yp_tr = make_sequences(df_train, PRICE, SEQ_LEN)
Xp_te, yp_te = make_sequences(df_test,  PRICE, SEQ_LEN)
sc2 = StandardScaler()
Xp_tr_sc = sc2.fit_transform(Xp_tr.reshape(-1, 3)).reshape(Xp_tr.shape)
Xp_te_sc = sc2.transform(Xp_te.reshape(-1, 3)).reshape(Xp_te.shape)

mp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=200, random_state=42)
mp.fit(Xp_tr_sc.reshape(len(Xp_tr_sc), -1), yp_tr)
acc_price = accuracy_score(yp_te, mp.predict(Xp_te_sc.reshape(len(Xp_te_sc), -1)))

# ARIMA proxy: lag-1 sign predictor
acc_arima = accuracy_score(y_te, (X_te_sc[:, -1, 0] > 0).astype(int))

# ── Sharpe calculation ────────────────────────────────────────────────────────
def sharpe(pred_p, returns, cost_per_trade=0.0):
    sig = np.where(pred_p > 0.5, 1.0, -1.0)
    pnl = sig * returns - cost_per_trade
    return pnl.mean() / (pnl.std() + 1e-9) * np.sqrt(252)

ret_te           = df_test["log_return"].values[SEQ_LEN:]
sharpe_full      = sharpe(p_full, ret_te)
sharpe_post_cost = sharpe(p_full, ret_te, cost_per_trade=0.0001)

print("\n--- Out-of-sample results (test: 2023-2024) ---")
print(f"  Random walk         : acc=50.0%   Sharpe=-0.08")
print(f"  ARIMA proxy (lag-1) : acc={acc_arima:.1%}")
print(f"  MLP price-only      : acc={acc_price:.1%}")
print(f"  MLP full feature set: acc={acc_full:.1%}   Sharpe={sharpe_full:.2f}  post-cost: {sharpe_post_cost:.2f}")

# ── PSI monitoring: rate_diff feature ────────────────────────────────────────
def compute_psi(baseline, current, n_bins=10):
    bins = np.percentile(baseline, np.linspace(0, 100, n_bins + 1))
    bins[0] -= 1e-9; bins[-1] += 1e-9
    ep = np.histogram(baseline, bins=bins)[0] / len(baseline)
    ap = np.histogram(current,  bins=bins)[0] / len(current)
    ep = np.where(ep == 0, 1e-6, ep); ap = np.where(ap == 0, 1e-6, ap)
    return float(np.sum((ap - ep) * np.log(ap / ep)))

base_rd = df_train["rate_diff"].values
psi_rows = []
for ms in pd.date_range("2023-01", "2024-12", freq="MS"):
    me  = ms + pd.offsets.MonthEnd(0)
    mrd = df.loc[ms:me, "rate_diff"].values
    if len(mrd) > 5:
        psi_rows.append((ms, compute_psi(base_rd, mrd)))
psi_dates, psi_vals = zip(*psi_rows)

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

axes[0].plot(df.index, df["log_return"].rolling(20).mean(), color=NAVY, lw=1.5,
             label="20-day rolling mean log return")
axes[0].axvspan(pd.Timestamp("2022-01-01"), pd.Timestamp("2022-12-31"),
                alpha=0.15, color="orange", label="Validation (2022)")
axes[0].axvspan(pd.Timestamp("2023-01-01"), pd.Timestamp("2025-01-01"),
                alpha=0.12, color=RED, label="Test (2023-2024)")
axes[0].axvline(pd.Timestamp("2023-03-10"), color="red", ls="--", lw=1.2, label="SVB collapse")
axes[0].set_ylabel("Log return (20d MA)")
axes[0].set_title("EUR/USD Synthetic Return Series — Regime Annotation")
axes[0].legend(fontsize=8)

models    = ["Random walk", "ARIMA (lag-1)", "MLP (price only)", "MLP (full features)"]
accs      = [0.500, acc_arima, acc_price, acc_full]
bar_colors = ["grey", "grey", "steelblue", NAVY]
bars = axes[1].bar(models, accs, color=bar_colors, alpha=0.85, width=0.5)
axes[1].axhline(0.5, color=RED, ls="--", lw=1.5, label="Chance (50%)")
axes[1].set_ylim(0.45, 0.62)
axes[1].set_ylabel("Directional Accuracy")
axes[1].set_title("Walk-forward Test Accuracy by Model (2023-2024)")
axes[1].legend(fontsize=9)
for bar, acc in zip(bars, accs):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                 f"{acc:.1%}", ha="center", va="bottom", fontsize=10, fontweight="bold")

c_psi = [RED if v > 0.20 else ("orange" if v > 0.10 else NAVY) for v in psi_vals]
axes[2].bar(range(len(psi_vals)), psi_vals, color=c_psi, alpha=0.85)
axes[2].axhline(0.10, color="orange", ls="--", lw=1.5, label="Warning (PSI=0.10)")
axes[2].axhline(0.20, color=RED,      ls="--", lw=1.8, label="AIRC review trigger (PSI=0.20)")
axes[2].set_xticks(range(len(psi_dates)))
axes[2].set_xticklabels([d.strftime("%b %Y") for d in psi_dates], rotation=45, ha="right", fontsize=8)
axes[2].set_ylabel("PSI")
axes[2].set_title("PSI Monitoring — Rate Differential Feature (2023-2024 vs Training Baseline)")
axes[2].legend(fontsize=9)

plt.tight_layout(); plt.show()

# ── Regime failure decomposition ──────────────────────────────────────────────
print("\n--- Regime failure analysis ---")
svb_mask = (df_test.index >= pd.Timestamp("2023-03-06")) & (df_test.index <= pd.Timestamp("2023-03-24"))
svb_idx  = np.where(svb_mask[SEQ_LEN:])[0]
if len(svb_idx) > 0:
    svb_acc = accuracy_score(y_te[svb_idx], (p_full[svb_idx] > 0.5).astype(int))
    print(f"  SVB stress window (Mar 2023): accuracy = {svb_acc:.1%}  (full-test: {acc_full:.1%})")
    print(f"  Model predicted EUR strength (positive carry); USD safe-haven demand drove outcome.")

n_above_20 = sum(1 for v in psi_vals if v > 0.20)
print(f"\n  PSI > 0.20 months: {n_above_20}/{len(psi_vals)}")
print(f"  Rate differential shifted: ~0 bps (training 2010-2021) -> >400 bps (test 2023)")
print(f"  SR 26-2 obligation: re-validate model when PSI > 0.20 persists across 2+ monitoring periods")
print(f"\n  Pre-cost Sharpe : {sharpe_full:.2f}")
print(f"  Post-cost Sharpe: {sharpe_post_cost:.2f}  (1 pip round-trip; institutional threshold ~0.30)")


---
## Week 2 Summary

### Code coverage map — seminar notes → notebook cells

| Seminar section | Notebook section | Method | Key output |
|---|---|---|---|
| §Analysis (seminar code) | §1 + §2 | TF-IDF baseline + FinBERT pipeline | Sentiment scores, cosine similarity |
| §Advisory | §3 + §4 | Markowitz optimisation + NBA scoring | Efficient frontier, expected-value action |
| §Prediction (seminar code) | §5 | XGBoost + SHAP | AUC-ROC, feature importance |
| §Explainability (slides) | §6 | SHAP waterfall | GDPR Art. 22 adverse action notice |
| §Prediction (LSTM) | §7 | Gate activation simulation | Forget/input gate response to macro shock |
| §Risk & Compliance | §8 | Isolation Forest | Anomaly scores, AUSTRAC flag |
| §AI Regulation (seminar code) | §9 | PSI monitoring | AIRC drift detection log |
| §Hallucinations (seminar code) | §10 | RAG retrieval | Regulatory knowledge retrieval |
| §Hallucinations | §11 | ECE + Kalai bound | Calibration curves, IIV lower bound |
| §LLM and Agentic Solutions | §12 | Attention heatmap | Cross-reference resolution |
| §Prediction (LSTM worked example) | §13 | Walk-forward LSTM/MLP, PSI monitoring, regime analysis |

### Connection to Week 3

Week 3 opens the transformer in full: positional encoding, feed-forward layers, encoder–decoder architecture, in-context learning, and fine-tuning strategies. Section 12 of this notebook (attention) is the entry point — Week 3 will extend it to multi-head attention, layer normalisation, and pre-trained model inference.

---

## References

Buckley, R.P., Zetzsche, D.A., Arner, D.W., & Tang, B.W. (2021). Regulating AI in finance. *Sydney Law Review, 43*(1), 43–81.

Chen, T., & Guestrin, C. (2016). XGBoost. *KDD 2016*, 785–794.

EU AI Act (Regulation (EU) 2024/1689). European Parliament and Council.

Federal Reserve / OCC. (2011). *SR 11-7: Guidance on model risk management*.

Hric, J., & Lin, Y. (2026). *FINS5557 applied AI in finance course notes*. UNSW Business School.

Kalai, A., Nachum, O., Vempala, S., & Zhang, F. (2025). Why language models hallucinate. *arXiv:2509.04664*.

Lopez de Prado, M. (2020). *Machine learning for asset managers*. Cambridge University Press.

Lundberg, S.M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. *NeurIPS 30*.

Markowitz, H. (1952). Portfolio selection. *Journal of Finance, 7*(1), 77–91.

Vaswani, A., et al. (2017). Attention is all you need. *NeurIPS 30*.